# 07 - Adapter merging: factor-space and ΔW controls  

This notebook creates merged LoRA adapters using different model merging strategies.  

In [3]:
import exp_common as E
import pandas as pd

RUN_SEEDS = [42, 123, 777]
ALPHA = 1.0
TOP_K = 0.2
DROP_RATE = 0.5
TARGET_RANK = 16
METHODS = ["task_arithmetic", "ties", "model_soup", "dare+task_arithmetic", "dare+ties", "dare+model_soup"]

audit_rows = []

def merge_set(seed, scenario, adapters, eval_anchor=None):
    for space in ["factor", "delta_w"]:
        for method in METHODS:
            safe = method.replace("+", "_plus_")
            out = E.MERGED_DIR / scenario / space / safe / f"seed_{seed}"
            E.merge_adapters(
                adapters, out, method=method, seed=seed, alpha=ALPHA,
                top_k=TOP_K, drop_rate=DROP_RATE, space=space, target_rank=TARGET_RANK
            )
            audit_rows.append(E.merge_operator_audit(
                adapters, out, method=method, space=space,
                top_k=TOP_K, drop_rate=DROP_RATE, seed=seed
            ) | {"scenario": scenario, "anchor": str(eval_anchor) if eval_anchor else None})
            print("saved", out)

for seed in RUN_SEEDS:
    logs = E.ADAPTERS_DIR / "logs" / f"seed_{seed}" / "adapter"
    qa = E.ADAPTERS_DIR / "qa" / f"seed_{seed}" / "adapter"
    merge_set(seed, "base_logs_plus_qa", [logs, qa])

    cpt = E.ADAPTERS_DIR / "cpt" / f"seed_{seed}" / "adapter"
    logs_cpt = E.ADAPTERS_DIR / "logs_on_cpt" / f"seed_{seed}" / "adapter"
    qa_cpt = E.ADAPTERS_DIR / "qa_on_cpt" / f"seed_{seed}" / "adapter"
    if all((p / "adapter_model.safetensors").exists() for p in [cpt, logs_cpt, qa_cpt]):
        merge_set(seed, "cpt_anchor_logs_plus_qa", [logs_cpt, qa_cpt], eval_anchor=cpt)

pd.DataFrame(audit_rows).to_csv(E.ANALYSIS_DIR / "merge_operator_audit.csv", index=False)
print("Merge experiments and operator audit complete")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
saved /notebooks/artifacts/merged_adapters/base_logs_plus_qa/factor/task_arithmetic/seed_42
saved /notebooks/artifacts/merged_adapters/base_logs_plus_qa/factor/ties/seed_42
saved /notebooks/artifacts/merged_adapters/base_logs_plus_qa/factor/model_soup/seed_42
saved /notebooks/artifacts/merged_adapters/base_logs_plus_qa/factor/dare_plus_task_arithmetic/seed_42
saved /notebooks/artifacts/merged_adapters/base_logs_plus_qa/factor/dare_plus_ties/seed_42
saved /notebooks/artifacts/merged_adapters/base_logs_plus_qa/factor/dare_plus_model_soup/seed_42
saved /notebooks/artifacts/merged_adapters/base_logs_plus_qa/delta_w/task_arithmetic/seed_42
saved /notebooks/artifacts/merged_adapters/base_logs_plus_qa/delta_w/ties/seed_42
saved /notebooks/artifacts/merged_adapters/base_logs_plus_qa/delta_w/model_soup/seed_42
saved /notebooks/artifacts/merged_adapters/base_l

In [ ]:
import exp_common as E
import shutil

for seed in [42, 123, 777]:
    cpt = E.ADAPTERS_DIR / "cpt" / f"seed_{seed}" / "adapter"
    logs = E.ADAPTERS_DIR / "logs_on_cpt" / f"seed_{seed}" / "adapter"
    qa = E.ADAPTERS_DIR / "qa_on_cpt" / f"seed_{seed}" / "adapter"
    if not all((p / "adapter_model.safetensors").exists() for p in [cpt, logs, qa]):
        continue
    for first, second, label in [(logs, qa, "logs_then_qa"), (qa, logs, "qa_then_logs")]:
        out = E.MERGED_DIR / "sequential_cpt_anchor" / "factor" / "task_arithmetic" / f"seed_{seed}" / label
        E.merge_adapters([first, second], out, method="task_arithmetic", alpha=1.0, seed=seed, space="factor")
        print("saved", out)

saved /notebooks/artifacts/merged_adapters/sequential_cpt_anchor/factor/task_arithmetic/seed_42/logs_then_qa
saved /notebooks/artifacts/merged_adapters/sequential_cpt_anchor/factor/task_arithmetic/seed_42/qa_then_logs
saved /notebooks/artifacts/merged_adapters/sequential_cpt_anchor/factor/task_arithmetic/seed_123/logs_then_qa
saved /notebooks/artifacts/merged_adapters/sequential_cpt_anchor/factor/task_arithmetic/seed_123/qa_then_logs
saved /notebooks/artifacts/merged_adapters/sequential_cpt_anchor/factor/task_arithmetic/seed_777/logs_then_qa
saved /notebooks/artifacts/merged_adapters/sequential_cpt_anchor/factor/task_arithmetic/seed_777/qa_then_logs
